# Nettoyage et préparation

Objectif : partir du dataset brut (vu dans `01_exploration.ipynb`) et produire un
fichier propre et exploitable dans `data/processed/games_clean.csv`.

Étapes : suppression des colonnes inutiles, gestion des doublons et des valeurs
manquantes, reformatage des types, création de quelques features utiles pour
la suite (ratio d'avis positifs, âge du jeu, nombre de genres/tags).

In [21]:
import pandas as pd
import numpy as np
import re
import ast

pd.set_option('display.max_columns', None)

# En-tête corrigé : le fichier brut a un bug ('DiscountDLC count' au lieu de
# 'Discount' + 'DLC count'), voir le notebook 01 pour le détail du problème.
COLONNES = [
    'AppID', 'Name', 'Release date', 'Estimated owners', 'Peak CCU',
    'Required age', 'Price', 'Discount', 'DLC count', 'About the game',
    'Supported languages', 'Full audio languages', 'Reviews', 'Header image',
    'Website', 'Support url', 'Support email', 'Windows', 'Mac', 'Linux',
    'Metacritic score', 'Metacritic url', 'User score', 'Positive', 'Negative',
    'Score rank', 'Achievements', 'Recommendations', 'Notes',
    'Average playtime forever', 'Average playtime two weeks',
    'Median playtime forever', 'Median playtime two weeks', 'Developers',
    'Publishers', 'Categories', 'Genres', 'Tags', 'Screenshots', 'Movies'
]

df = pd.read_csv('../data/raw/games.csv', skiprows=1, names=COLONNES, low_memory=False)
print(f"Dimensions avant nettoyage : {df.shape[0]} lignes, {df.shape[1]} colonnes")

Dimensions avant nettoyage : 125855 lignes, 40 colonnes


## 1. Suppression des colonnes inutiles

On retire les colonnes qui ne servent ni à l'analyse ni au modèle : descriptions
longues, URLs (images, site web, support), notes libres. On vérifie d'abord
quelles colonnes existent vraiment dans le fichier avant de les supprimer, pour
éviter une erreur si une colonne a été renommée ou n'existe pas dans ta version
du fichier.

In [22]:
colonnes_a_supprimer = [
    'About the game', 'Detailed description', 'Short description',
    'Header image', 'Website', 'Support url', 'Support email',
    'Screenshots', 'Movies', 'Reviews', 'Notes', 'Metacritic url',
    'Score rank'
]

colonnes_presentes = [c for c in colonnes_a_supprimer if c in df.columns]
colonnes_absentes = [c for c in colonnes_a_supprimer if c not in df.columns]

df = df.drop(columns=colonnes_presentes)

print(f"Colonnes supprimées ({len(colonnes_presentes)}) : {colonnes_presentes}")
if colonnes_absentes:
    print(f"Colonnes non trouvées dans ton fichier (ignorées) : {colonnes_absentes}")

Colonnes supprimées (11) : ['About the game', 'Header image', 'Website', 'Support url', 'Support email', 'Screenshots', 'Movies', 'Reviews', 'Notes', 'Metacritic url', 'Score rank']
Colonnes non trouvées dans ton fichier (ignorées) : ['Detailed description', 'Short description']


## 2. Doublons

On supprime les lignes strictement identiques, puis on vérifie qu'il n'y a pas
deux lignes avec le même `AppID` (l'identifiant unique d'un jeu sur Steam).

In [23]:
nb_doublons_avant = df.duplicated().sum()
df = df.drop_duplicates()

if 'AppID' in df.columns:
    nb_doublons_id = df.duplicated(subset='AppID').sum()
    df = df.drop_duplicates(subset='AppID', keep='first')
    print(f"Doublons exacts supprimés : {nb_doublons_avant}")
    print(f"Doublons sur AppID supprimés : {nb_doublons_id}")

print(f"Dimensions après suppression des doublons : {df.shape[0]} lignes")

Doublons exacts supprimés : 0
Doublons sur AppID supprimés : 0
Dimensions après suppression des doublons : 125855 lignes


## 3. Valeurs manquantes sur les colonnes essentielles

On supprime les lignes sans nom ou sans date de sortie (inutilisables pour
l'analyse), et on remplace les développeurs/éditeurs manquants par 'Unknown'
plutôt que de perdre la ligne.

In [24]:
avant = len(df)
df = df.dropna(subset=['Name', 'Release date'])
print(f"Lignes supprimées car Name ou Release date manquant : {avant - len(df)}")

for col in ['Developers', 'Publishers']:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')

Lignes supprimées car Name ou Release date manquant : 1


## 4. Reformatage des types

`Release date` devient une vraie date, dont on extrait l'année. `Estimated
owners` est une fourchette texte (ex : "0 - 20000") qu'on convertit en valeur
numérique en prenant le milieu de la fourchette, pour pouvoir l'utiliser dans
des calculs et plus tard dans un modèle.

In [25]:
df['Release date'] = pd.to_datetime(df['Release date'], errors='coerce')
df['release_year'] = df['Release date'].dt.year

nb_dates_invalides = df['Release date'].isna().sum()
print(f"Dates non reconnues après conversion : {nb_dates_invalides}")

Dates non reconnues après conversion : 0


In [26]:
def parse_estimated_owners(valeur):
    """Transforme '0 - 20000' en 10000.0 (la moyenne de la fourchette)."""
    if pd.isna(valeur):
        return np.nan
    nombres = re.findall(r'\d+', str(valeur).replace(',', ''))
    if len(nombres) == 2:
        bas, haut = int(nombres[0]), int(nombres[1])
        return (bas + haut) / 2
    return np.nan

if 'Estimated owners' in df.columns:
    df['estimated_owners_avg'] = df['Estimated owners'].apply(parse_estimated_owners)
    print(df[['Estimated owners', 'estimated_owners_avg']].head())

  Estimated owners  estimated_owners_avg
0            0 - 0                   0.0
1        0 - 20000               10000.0
2        0 - 20000               10000.0
3        0 - 20000               10000.0
4        0 - 20000               10000.0


## 5. Features pour l'analyse

On crée :
- `total_reviews` et `positive_ratio` (avis positifs / total des avis), la
  variable qu'on utilisera probablement comme cible pour le modèle plus tard
- `game_age_years`, l'âge du jeu par rapport au jeu le plus récent du dataset
- `nb_genres`, `nb_categories`, `nb_tags`, le nombre d'éléments dans ces colonnes
- `main_genre`, le premier genre listé pour le jeu

In [27]:
if {'Positive', 'Negative'}.issubset(df.columns):
    df['total_reviews'] = df['Positive'] + df['Negative']
    df['positive_ratio'] = np.where(
        df['total_reviews'] > 0,
        df['Positive'] / df['total_reviews'],
        np.nan
    )

annee_reference = df['release_year'].max()
df['game_age_years'] = annee_reference - df['release_year']

In [28]:
def compter_elements(valeur):
    """Compte le nombre d'éléments dans une colonne liste/dict/texte,
    quel que soit le format exact (liste Python en texte, ou valeurs
    séparées par des virgules)."""
    if pd.isna(valeur):
        return 0
    texte = str(valeur).strip()
    try:
        parsed = ast.literal_eval(texte)
        if isinstance(parsed, (list, dict, set, tuple)):
            return len(parsed)
    except (ValueError, SyntaxError):
        pass
    return len([x for x in texte.split(',') if x.strip()])


def premier_element(valeur):
    """Renvoie le premier genre/tag listé, ou 'Unknown' si rien d'exploitable."""
    if pd.isna(valeur):
        return 'Unknown'
    texte = str(valeur).strip()
    try:
        parsed = ast.literal_eval(texte)
        if isinstance(parsed, dict):
            return list(parsed.keys())[0] if parsed else 'Unknown'
        if isinstance(parsed, (list, tuple, set)):
            return list(parsed)[0] if parsed else 'Unknown'
    except (ValueError, SyntaxError):
        pass
    parts = [x.strip() for x in texte.split(',') if x.strip()]
    return parts[0] if parts else 'Unknown'


for col, nom in [('Genres', 'genres'), ('Categories', 'categories'), ('Tags', 'tags')]:
    if col in df.columns:
        df[f'nb_{nom}'] = df[col].apply(compter_elements)

if 'Genres' in df.columns:
    df['main_genre'] = df['Genres'].apply(premier_element)

## 6. Valeurs potentiellement aberrantes (à garder en tête, pas supprimées ici)

On les repère mais on ne les supprime pas : elles peuvent rester pertinentes
pour l'analyse (un jeu gratuit n'est pas une erreur). On filtrera si besoin au
moment de construire le modèle (Jour 5), en particulier les jeux sans aucun
avis pour qui `positive_ratio` est manquant.

In [29]:
if 'Price' in df.columns:
    print(f"Jeux gratuits (Price = 0) : {(df['Price'] == 0).sum()}")

if 'total_reviews' in df.columns:
    print(f"Jeux sans aucun avis : {(df['total_reviews'] == 0).sum()}")

Jeux gratuits (Price = 0) : 26660
Jeux sans aucun avis : 42898


## 7. Sauvegarde du fichier propre

In [30]:
chemin_sortie = '../data/processed/games_clean.csv'
df.to_csv(chemin_sortie, index=False)

print(f"Fichier sauvegardé : {chemin_sortie}")
print(f"Dimensions finales : {df.shape[0]} lignes, {df.shape[1]} colonnes")

Fichier sauvegardé : ../data/processed/games_clean.csv
Dimensions finales : 125854 lignes, 38 colonnes


## Notes et prochaines étapes

- Comparer le nombre de lignes/colonnes avant et après nettoyage (à reporter
  dans le notebook ou la doc, c'est une info utile pour l'oral)
- Vérifier la distribution de `estimated_owners_avg` et `positive_ratio`
  (probablement très asymétriques, prévoir une échelle log pour les graphiques)
- Étudier `main_genre`, `release_year`, `Price` comme variables explicatives
  potentielles du succès d'un jeu